# Ghostexec — Unsloth + TRL SFT -> GRPO against the deployed HF Space API

Post-train `unsloth/Llama-3.2-3B-Instruct` with **SFT warmup first** and then GRPO, where rewards are fetched over HTTP from the **live** Ghostexec OpenEnv Space.

- Live endpoint: `https://modelbuilderhq-ghostexec.hf.space`
- Algorithm: TRL `0.22.2` `SFTTrainer` -> `GRPOTrainer` (no vLLM — HF `generate()` path)
- Base (recommended for fast winning iterations): `unsloth/Qwen2.5-3B-Instruct` (4-bit) + LoRA r=16 + bf16
- Curriculum: **easy -> full** annealing (strong local scaffold early, env-dominant later)
- Rewards: four **independent** functions — `env_reward` (live Space) / `format_reward` / `semantic_action_reward` / `anti_idle_reward`

### Help Guide phase map (notebook sections mirror `[Participant Help Guide] §18`)
| Phase | Where |
|---|---|
| 1 Pick a narrow task | section 1 |
| 2 Build the environment | section 2 (already deployed; health check here) |
| 3 Build rewards | section 3 |
| 4 Deploy | section 4 (confirm) |
| 5 Train small | section 5 (SFT + Stage B) |
| 6 Inspect for hacking | section 6 |
| 7 Add curriculum | section 7 (Stages C + D) |
| 8 Train bigger | section 8 (knobs, not action) |
| 9 Save and demo | section 9 |

## Phase 1 — Pick a narrow task

Single-step action selection from a plain-text executive briefing. The model reads the briefing from `/reset` and must emit exactly one JSON action matching `GhostexecAction`. The deployed Space scores that action and returns a reward from `/step`. That reward is the learning signal.

Legal `action_type` values: `reply_email, archive_email, reschedule_meeting, cancel_meeting, complete_task, delegate_task, send_message, do_nothing`.

The scenario is fixed on the deployed Space (`phase2_core`), so the curriculum is an **exploration schedule** (temperature / num_generations / learning rate) across three training stages rather than a scenario switch.

## Phase 2 — Build the environment (already deployed on HF Spaces)

The next cell is the exact Unsloth install snippet. Restart the runtime after it finishes if Colab asks you to.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
%pip install -q requests pydantic matplotlib pandas tqdm huggingface_hub datasets
print("aux deps installed")

In [ ]:
import os, sys, json, time, random, re, math, pathlib
from typing import Any

GHOSTEXEC_ENV_URL = os.environ.get("GHOSTEXEC_ENV_URL", "https://modelbuilderhq-ghostexec.hf.space")
# Small-model-first default for rapid iteration and higher success probability.
MODEL_ID          = os.environ.get("MODEL_ID", "unsloth/Qwen2.5-3B-Instruct")
RUN_NAME          = os.environ.get("RUN_NAME", "ghostexec-unsloth-grpo")
HUB_REPO_ID       = os.environ.get("HUB_REPO_ID", "")
OUT = pathlib.Path("/content/ghostexec_out") if os.path.exists("/content") else pathlib.Path("./ghostexec_out")
OUT.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata  # type: ignore
    if not os.environ.get("HF_TOKEN"):
        try: os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
        except Exception: pass
except Exception:
    pass

print("Endpoint :", GHOSTEXEC_ENV_URL)
print("Model    :", MODEL_ID)
print("Output   :", OUT)
print("HF token :", "set" if os.environ.get("HF_TOKEN") else "missing (needed only for push_to_hub)")

### 2.1 HTTP client to the deployed Space

Every reward in this notebook comes from this class — we never run Ghostexec locally.

In [ ]:
import requests

class GhostexecSpace:
    def __init__(self, url: str, timeout: float = 60.0, max_retries: int = 4):
        self.url = url.rstrip("/")
        self.timeout = timeout
        self.max_retries = max_retries
        self.latency_ms: list[float] = []

    def _post(self, path: str, payload: dict) -> dict:
        last_err: Exception | None = None
        for attempt in range(self.max_retries):
            try:
                t0 = time.perf_counter()
                r = requests.post(f"{self.url}{path}", json=payload, timeout=self.timeout)
                self.latency_ms.append((time.perf_counter() - t0) * 1000.0)
                r.raise_for_status()
                return r.json()
            except Exception as e:
                last_err = e
                time.sleep(min(2 ** attempt, 8.0))
        raise RuntimeError(f"POST {path} failed after {self.max_retries} tries: {last_err}")

    def reset(self) -> dict:
        return self._post("/reset", {})

    def step(self, action: dict) -> tuple[float, dict]:
        raw = self._post("/step", {"action": action})
        reward = raw.get("reward")
        if reward is None:
            reward = (raw.get("observation") or {}).get("reward", 0.0)
        try:    return float(reward), raw
        except Exception: return 0.0, raw

env = GhostexecSpace(GHOSTEXEC_ENV_URL)
print("Health reset ...")
_obs = env.reset()
print("reset keys:", sorted(_obs.keys()))
_brief = ((_obs.get("observation") or _obs).get("echoed_message") or "")[:400]
print("briefing preview:\n", _brief)

### 2.2 Verifier sanity check (Help Guide §8)

**Colab / stale cells:** If the traceback mentions **`do_nothing is not the worst/floor`** on **line ~28**, you are running **old cached notebook code** (that assert was removed). Use **Runtime → Disconnect and delete runtime**, then **re-clone** the repo or **re-download** this notebook from GitHub and run from the top.

**If every proactive action prints `-0.25` and only `do_nothing` is `-0.15`:** every non-idle smoke is an **invalid step** (wrong ids like `email_01`, or an outdated `_smoke_action`). This cell expects **real `phase2_core` ids** (`e01`, `e09`, `m02`, …) — see `_smoke_action` below.

Fire every legal `action_type` once with **semantically valid** payloads (real ids from `scenarios/phase2_core.json`). Fake ids deserialize but fail validation (−0.25 invalid-step) and are not a fair probe. Also: **`do_nothing` is not guaranteed to be the lowest reward** — a valid but harmful action (e.g. cancelling an important meeting) can push the weighted score below the idle penalty. We instead assert **non-idle smokes are `step_ok=True`** and **`do_nothing` scores below a benign `reply_email` on `e01`**. If rewards are all identical, abort — GRPO cannot learn from a degenerate verifier.

In [ ]:
LEGAL_ACTION_TYPES = [
    "reply_email", "archive_email", "reschedule_meeting", "cancel_meeting",
    "complete_task", "delegate_task", "send_message", "do_nothing",
]

def _smoke_action(action_type: str) -> dict:
    # Real IDs from phase2_core scenario
    base = {"action_type": action_type, "message": ""}

    if action_type == "reply_email":
        return {**base, "email_id": "e01", "message_body": "Acknowledged — on it now."}
    if action_type == "archive_email":
        return {**base, "email_id": "e09"}
    if action_type == "reschedule_meeting":
        return {
            **base,
            "meeting_id": "m02",
            "new_time": "2026-04-21T18:00:00",
            "reason": "freeing the morning block",
        }
    if action_type == "cancel_meeting":
        return {**base, "meeting_id": "m10", "reason": "smoke test cancel"}
    if action_type == "complete_task":
        return {**base, "task_id": "t07"}
    if action_type == "delegate_task":
        return {**base, "task_id": "t08", "contact_name": "Jordan Lee"}
    if action_type == "send_message":
        return {
            **base,
            "contact_name": "Jamie Liu",
            "message_body": "Quick sync when you have a minute.",
        }

    # do_nothing
    return {
        **base,
        "email_id": "",
        "message_body": "",
        "meeting_id": "",
        "new_time": "",
        "reason": "",
        "task_id": "",
        "contact_name": "",
    }

rewards_by_action: dict[str, float] = {}
step_ok_by_action: dict[str, bool | None] = {}

for at in LEGAL_ACTION_TYPES:
    env.reset()
    r, raw = env.step(_smoke_action(at))
    rewards_by_action[at] = round(r, 4)
    obs = raw.get("observation") or {}
    step_ok_by_action[at] = (obs.get("metadata") or {}).get("step_ok")

print(json.dumps({"reward": rewards_by_action, "step_ok": step_ok_by_action}, indent=2))

uniq = set(rewards_by_action.values())
assert len(uniq) > 1, "Verifier is constant across actions — env can't teach anything."

# All non-idle smokes must be valid
for at in LEGAL_ACTION_TYPES:
    if at == "do_nothing":
        continue
    assert step_ok_by_action.get(at) is True, f"{at} smoke is invalid; check IDs."

# Idle should be worse than benign good action
assert rewards_by_action["do_nothing"] < rewards_by_action["reply_email"] - 1e-6, \
    "do_nothing should score below reply_email(e01)."

print("\nverifier OK — rewards vary, smokes are valid, do_nothing < reply_email(e01).")

## Phase 3 — Build rewards

Three independent reward functions per Help Guide §7. Keeping them independent means we can plot each component, watch their correlations, and catch hacking (e.g. env reward climbs while format reward collapses).

In [ ]:
from pydantic import BaseModel
from typing import Literal

GhostexecActionType = Literal[
    "reply_email", "archive_email", "reschedule_meeting", "cancel_meeting",
    "complete_task", "delegate_task", "send_message", "do_nothing",
]

class GhostexecAction(BaseModel):
    action_type:   GhostexecActionType = "do_nothing"
    email_id:      str = ""
    message_body:  str = ""
    meeting_id:    str = ""
    new_time:      str = ""
    reason:        str = ""
    task_id:       str = ""
    contact_name: str = ""
    message:       str = ""

def _extract_json(text: str) -> dict:
    s = text.strip()
    s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.IGNORECASE | re.MULTILINE).strip()
    start, end = s.find("{"), s.rfind("}")
    if start == -1 or end <= start: raise ValueError("no json object")
    return json.loads(s[start:end+1])

def parse_action_strict(text: str) -> dict:
    obj = _extract_json(text)
    GhostexecAction(**obj)
    return obj

def parse_action(text: str) -> dict:
    try: return parse_action_strict(text)
    except Exception: return {"action_type": "do_nothing"}

LEGAL_ACTION_TYPES = {
    "reply_email", "archive_email", "reschedule_meeting", "cancel_meeting",
    "complete_task", "delegate_task", "send_message", "do_nothing",
}
LEGAL_ACTION_KEYS = {
    "action_type", "email_id", "message_body", "meeting_id",
    "new_time", "reason", "task_id", "contact_name", "message",
}


def sanitize_action(raw: dict) -> dict:
    """Keep only legal Ghostexec fields and coerce malformed IDs/actions safely."""
    action = {k: v for k, v in (raw or {}).items() if k in LEGAL_ACTION_KEYS}

    at = str(action.get("action_type", "do_nothing"))
    if at not in LEGAL_ACTION_TYPES:
        at = "do_nothing"
    action["action_type"] = at

    # Common model mistake: writes message text into `message` instead of `message_body`.
    if at in {"reply_email", "send_message"}:
        if not action.get("message_body") and action.get("message"):
            action["message_body"] = action["message"]

    if "email_id" in action and not re.fullmatch(r"e\d{2}", str(action["email_id"])):
        action["email_id"] = ""
    if "meeting_id" in action and not re.fullmatch(r"m\d{2}", str(action["meeting_id"])):
        action["meeting_id"] = ""
    if "task_id" in action and not re.fullmatch(r"t\d{2}", str(action["task_id"])):
        action["task_id"] = ""

    return action

assert parse_action_strict('```json\n{"action_type":"archive_email","email_id":"email_01"}\n```')["action_type"] == "archive_email"
assert parse_action("garbage")["action_type"] == "do_nothing"
print("parser OK")

In [ ]:
def _completion_text(c) -> str:
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return c[0].get("content", "")
    return c if isinstance(c, str) else str(c)


def _prompt_to_text(p) -> str:
    if isinstance(p, list) and p and isinstance(p[-1], dict):
        return str(p[-1].get("content", ""))
    if isinstance(p, dict):
        return str(p.get("content", ""))
    return str(p)


# Curriculum scalars are updated per stage: easy -> full.
CURRENT_ENV_SCALE = 0.85
CURRENT_LOCAL_SCALE = 0.60


def env_reward(completions, prompts=None, **_) -> list[float]:
    out: list[float] = []
    for c in completions:
        text = _completion_text(c)
        action = sanitize_action(parse_action(text))
        try:
            env.reset()
            r, _ = env.step(action)
        except Exception:
            r = -1.0
        out.append(float(r) * CURRENT_ENV_SCALE)
    return out


def format_reward(completions, **_) -> list[float]:
    out: list[float] = []
    for c in completions:
        text = _completion_text(c)
        try:
            parse_action_strict(text)
            out.append(0.12 * CURRENT_LOCAL_SCALE)
        except Exception:
            out.append(-0.20 * CURRENT_LOCAL_SCALE)
    return out


def semantic_action_reward(completions, prompts=None, **_) -> list[float]:
    """
    Reward canonical, briefing-grounded action payloads before env call.
    Scaled by CURRENT_LOCAL_SCALE for easy->full curriculum annealing.
    """
    out: list[float] = []
    for i, c in enumerate(completions):
        text = _completion_text(c)
        act = parse_action(text)
        at = act.get("action_type", "do_nothing")

        prompt_text = ""
        if prompts is not None and i < len(prompts):
            prompt_text = _prompt_to_text(prompts[i])

        def present(tok: str) -> bool:
            return bool(tok) and re.search(rf"\b{re.escape(tok)}\b", prompt_text) is not None

        r = -0.30
        if at == "do_nothing":
            r = -0.05
        elif at == "reply_email":
            eid = act.get("email_id", "")
            mb = (act.get("message_body", "") or "").strip()
            r = 0.30 if present(eid) and bool(re.fullmatch(r"e\d{2}", eid)) and mb else -0.30
        elif at == "archive_email":
            eid = act.get("email_id", "")
            r = 0.30 if present(eid) and bool(re.fullmatch(r"e\d{2}", eid)) else -0.30
        elif at == "reschedule_meeting":
            mid = act.get("meeting_id", "")
            nt = (act.get("new_time", "") or "").strip()
            r = 0.30 if present(mid) and bool(re.fullmatch(r"m\d{2}", mid)) and nt else -0.30
        elif at == "cancel_meeting":
            mid = act.get("meeting_id", "")
            r = 0.30 if present(mid) and bool(re.fullmatch(r"m\d{2}", mid)) else -0.30
        elif at == "complete_task":
            tid = act.get("task_id", "")
            r = 0.30 if present(tid) and bool(re.fullmatch(r"t\d{2}", tid)) else -0.30
        elif at == "delegate_task":
            tid = act.get("task_id", "")
            cn = (act.get("contact_name", "") or "").strip()
            r = 0.30 if present(tid) and bool(re.fullmatch(r"t\d{2}", tid)) and (cn in prompt_text) else -0.30
        elif at == "send_message":
            cn = (act.get("contact_name", "") or "").strip()
            mb = (act.get("message_body", "") or "").strip()
            r = 0.30 if cn and (cn in prompt_text) and mb else -0.30

        out.append(float(r) * CURRENT_LOCAL_SCALE)
    return out


def anti_idle_reward(completions, **_) -> list[float]:
    out: list[float] = []
    for c in completions:
        text = _completion_text(c)
        act = parse_action(text)
        out.append((-0.28 if act.get("action_type") == "do_nothing" else 0.03) * CURRENT_LOCAL_SCALE)
    return out


_dummy = '{"action_type":"archive_email","email_id":"email_01"}'
print("env      :", env_reward([_dummy]))
print("format   :", format_reward([_dummy]))
print("semantic :", semantic_action_reward([_dummy], prompts=["... e01 e09 t07 m02 Jamie Liu ..."]))
print("anti_idle:", anti_idle_reward([_dummy]))

In [ ]:
# GRPO stages B / C / D: no early-stop callbacks — each stage runs the full `max_steps`.


## Phase 4 — Deploy

Already done. Live Space: [`modelbuilderhq/ghostexec`](https://huggingface.co/spaces/modelbuilderhq/ghostexec). The health-check cell above confirmed `/reset` + `/step` are green.

## Phase 5 — Train small (SFT warmup -> GRPO)

Load `unsloth/Llama-3.2-3B-Instruct` in 4-bit with Unsloth, attach LoRA, run a **short SFT warmup first**, then run GRPO. vLLM is not used anywhere in this notebook — rollouts go through the standard HF `generate()` path inside `GRPOTrainer`.

In [ ]:
# IMPORTANT: import unsloth before transformers so its kernels patch cleanly.
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

policy, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,                 # auto (bf16 on T4 compute via bnb)
)

policy = FastLanguageModel.get_peft_model(
    policy,
    r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("policy loaded:", MODEL_ID)
policy.print_trainable_parameters()

In [ ]:
SYSTEM_PROMPT = (
    "You are Ghostexec, an AI Chief of Staff. You receive a plain-text briefing of an executive's "
    "inbox, calendar and tasks. You must choose the single best next action.\n\n"
    "Legal action_type values: reply_email, archive_email, reschedule_meeting, cancel_meeting, "
    "complete_task, delegate_task, send_message, do_nothing.\n\n"
    "Output ONLY a compact JSON object with these keys (no prose, no code fences):\n"
    "{\"action_type\": \"\", \"email_id\": \"\", \"message_body\": \"\", "
    "\"meeting_id\": \"\", \"new_time\": \"\", \"reason\": \"\", \"task_id\": \"\", "
    "\"contact_name\": \"\", \"message\": \"\"}.\n\n"
    "ID RULES:\n"
    "- email_id must be an email token from briefing like e01, e02, ...\n"
    "- meeting_id must be a meeting token like m01, m02, ...\n"
    "- task_id must be a task token like t01, t02, ...\n"
    "- contact_name must exactly match a contact shown in briefing.\n"
    "- Never use subject/body/description text as an ID.\n"
    "- If you cannot find a valid ID for your chosen action, output {\"action_type\":\"do_nothing\"}.\n\n"
    "Prefer high-impact valid actions; avoid do_nothing when critical items are unresolved."
)

def build_prompt(briefing: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"BRIEFING:\n{briefing}\n\nReturn one JSON action."},
    ]

def render_chat(messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
from datasets import Dataset
from tqdm.auto import tqdm

def fetch_briefing() -> str:
    obs = env.reset()
    inner = obs.get("observation") or obs
    brief = inner.get("echoed_message") or inner.get("message") or ""
    if not brief:
        raise RuntimeError(f"Space returned no briefing: keys={list(inner.keys())}")
    return brief

N_BRIEFINGS = int(os.environ.get("N_BRIEFINGS", "24"))
briefings: list[str] = []
for _ in tqdm(range(N_BRIEFINGS), desc="sampling /reset"):
    briefings.append(fetch_briefing())

print(f"fetched {len(briefings)} briefings ({len(set(briefings))} unique)")
train_ds = Dataset.from_list([{"prompt": build_prompt(b)} for b in briefings])
print(train_ds)

### 5.1 Baselines — weak random vs frozen (pre-pipeline) vs trained (Help Guide §19)

- **Random**: junk actions (bad IDs / not in scenario) so the mean stays low on purpose.
- **Frozen**: same LoRA policy **before** the SFT+GRPO cells below; should beat random clearly.
- **Trained**: re-eval after GRPO; Phase 9 asserts **random mean < frozen mean** and **frozen mean + margin < trained mean** (margins: env vars `MIN_GAP_RANDOM_FROZEN`, `MIN_GAP_FROZEN_TRAINED`).

In [ ]:
N_EVAL = int(os.environ.get("N_EVAL", "8"))

def random_policy_reward() -> list[float]:
    rs: list[float] = []
    for _ in range(N_EVAL):
        at = random.choice(list(LEGAL_ACTION_TYPES))
        env.reset()
        r, _ = env.step(_smoke_action(at))
        rs.append(r)
    return rs

@torch.no_grad()
def evaluate_policy(model, n: int = N_EVAL, temperature: float = 0.2) -> list[float]:
    FastLanguageModel.for_inference(model)
    rs: list[float] = []
    for i in range(n):
        brief = briefings[i % len(briefings)]
        prompt_text = render_chat(build_prompt(brief))
        inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-5),
            pad_token_id=tokenizer.pad_token_id,
        )
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        action = sanitize_action(parse_action(completion))
        env.reset()
        try:
            r, _ = env.step(action)
        except RuntimeError:
            r, _ = env.step({"action_type": "do_nothing"})
        rs.append(r)
    FastLanguageModel.for_training(model)
    return rs

print("Random baseline ...")
random_rewards = random_policy_reward()
print(" mean:", sum(random_rewards) / len(random_rewards))

print("Frozen-base baseline ...")
frozen_rewards = evaluate_policy(policy, n=N_EVAL, temperature=0.2)
print(" mean:", sum(frozen_rewards) / len(frozen_rewards))

baselines = {"random": random_rewards, "frozen": frozen_rewards}

### 5.2 Stage B — first GRPO stage (easy->full curriculum starts here)

We run a short SFT warmup first, then GRPO Stage B with stronger local scaffold weights (`CURRENT_LOCAL_SCALE`) and slightly lower env scale (`CURRENT_ENV_SCALE`).

As stages progress (B -> C -> D), the notebook anneals toward full env-dominant training.

In [ ]:
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer

reward_funcs = [env_reward, format_reward, semantic_action_reward, anti_idle_reward]
stage_logs: dict[str, list[dict]] = {}

# -------- SFT warmup --------
def _heuristic_action_for_sft(briefing: str) -> dict:
    b = briefing.lower()
    if "e01" in b:
        return {"action_type": "reply_email", "email_id": "e01", "message_body": "Acknowledged, sharing an update shortly."}
    if "m02" in b:
        return {"action_type": "reschedule_meeting", "meeting_id": "m02", "new_time": "2026-04-21T18:00:00", "reason": "resolve overlap"}
    if "t06" in b:
        return {"action_type": "complete_task", "task_id": "t06"}
    return {"action_type": "do_nothing"}

sft_rows = []
for b in briefings:
    msgs = build_prompt(b)
    prompt_txt = render_chat(msgs)
    completion_txt = json.dumps(_heuristic_action_for_sft(b), ensure_ascii=True)
    sft_rows.append({"prompt_text": prompt_txt, "completion_text": completion_txt})

sft_ds = Dataset.from_list(sft_rows)
sft_cfg = SFTConfig(
    output_dir=str(OUT / "sft_warmup"),
    max_steps=30,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=5,
    report_to="none",
)
sft_trainer = SFTTrainer(
    model=policy,
    processing_class=tokenizer,
    train_dataset=sft_ds,
    args=sft_cfg,
    dataset_text_field="prompt_text",
    formatting_func=lambda ex: [f"{p}{c}" for p, c in zip(ex["prompt_text"], ex["completion_text"])],
)
print("\n=== SFT warmup ===")
sft_trainer.train()
policy = sft_trainer.model


def grpo_config(name: str, *, temperature: float, num_generations: int, max_steps: int, lr: float) -> GRPOConfig:
    return GRPOConfig(
        output_dir=str(OUT / f"stage_{name}"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=num_generations,
        max_prompt_length=1920,
        max_completion_length=48,
        temperature=temperature,
        learning_rate=lr,
        beta=0.04,
        max_steps=max_steps,
        logging_steps=1,
        bf16=False,
        fp16=True,
        report_to="none",
        save_strategy="no",
        remove_unused_columns=False,
        log_completions=True,
    )


def set_curriculum_scales(stage_name: str) -> None:
    global CURRENT_ENV_SCALE, CURRENT_LOCAL_SCALE
    # easy -> full complexity curriculum
    if stage_name == "B":
        CURRENT_ENV_SCALE = 0.85
        CURRENT_LOCAL_SCALE = 0.60
    elif stage_name == "C":
        CURRENT_ENV_SCALE = 0.95
        CURRENT_LOCAL_SCALE = 0.40
    else:
        CURRENT_ENV_SCALE = 1.00
        CURRENT_LOCAL_SCALE = 0.25
    print(f"curriculum[{stage_name}] env={CURRENT_ENV_SCALE:.2f} local={CURRENT_LOCAL_SCALE:.2f}")


def run_stage(name: str, **kw) -> None:
    set_curriculum_scales(name)
    print(f"\n=== Stage {name} → {kw} ===")
    trainer = GRPOTrainer(
        model=policy,
        args=grpo_config(name, **kw),
        train_dataset=train_ds,
        reward_funcs=reward_funcs,
        processing_class=tokenizer,
    )
    trainer.train()
    stage_logs[name] = list(trainer.state.log_history)
    adapter_dir = OUT / f"adapter_stage_{name}"
    trainer.model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"stage {name} adapter → {adapter_dir}")


run_stage("B", temperature=0.8, num_generations=2, max_steps=20, lr=5e-6)

## Phase 6 — Inspect for hacking

Don't trust the mean reward alone. Sample six post-Stage-B completions, parse them, hit the Space live, and print the full trio (completion / parsed action / reward). Look for obviously pathological outputs (repeated identical JSON, prose-only outputs, empty fields).

In [ ]:
FastLanguageModel.for_inference(policy)
for i in range(6):
    brief = briefings[i % len(briefings)]
    prompt_text = render_chat(build_prompt(brief))
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(policy.device)
    out = policy.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.7,
                         pad_token_id=tokenizer.pad_token_id)
    completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    act = parse_action(completion)
    env.reset(); r, _ = env.step(act)
    print(f"\n--- sample {i} ---")
    print("completion:", completion.strip()[:200])
    print("parsed    :", json.dumps(act))
    print("reward    :", round(r, 4))
FastLanguageModel.for_training(policy)

## Phase 7 — Add curriculum

The deployed Space scenario is fixed, so curriculum is applied through both:

1. **Exploration schedule** (temperature/lr across stages)
2. **Complexity curriculum (easy -> full)** via reward scales:
   - Stage B: stronger local scaffold guidance
   - Stage C: mixed guidance
   - Stage D: env-dominant optimization

In [ ]:
run_stage("C", temperature=0.7, num_generations=2, max_steps=25, lr=5e-6)
run_stage("D", temperature=0.5, num_generations=2, max_steps=15, lr=2e-6)

## Phase 8 — Train bigger (knobs, not action)

Only after the loop is stable should you scale. If you rent an L4 or A100 with HF credits:

- `MODEL_ID` → `unsloth/Qwen3-4B-Instruct-2507` or `unsloth/Llama-3.1-8B-Instruct`
- `N_BRIEFINGS` ↑ (more prompt diversity)
- `num_generations` ↑ and `max_steps` ↑ (more rollouts per prompt, more updates)

All other cells are unchanged. Don't add features until you've watched a full stable run on this small config.

## Phase 9 — Save and demo

Re-evaluate on the same `N_EVAL` prompts, plot the before/after + reward curves, save the LoRA adapter (no 4-bit merge per Help Guide §16), and write a compliance manifest.

In [ ]:
print("Evaluating trained policy ...")
trained_rewards = evaluate_policy(policy, n=N_EVAL, temperature=0.2)
print(" trained mean:", sum(trained_rewards) / len(trained_rewards))

def _mean(xs): return sum(xs) / max(len(xs), 1)
summary = {
    "random":  _mean(baselines["random"]),
    "frozen":  _mean(baselines["frozen"]),
    "trained": _mean(trained_rewards),
}
print(json.dumps(summary, indent=2))

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(list(summary.keys()), list(summary.values()), color=["#888", "#1f77b4", "#2ca02c"])
plt.title("Ghostexec: mean reward vs deployed HF Space")
plt.ylabel("mean episode reward (higher is better)")
plt.axhline(0.0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig(OUT / "before_after.png", dpi=150)
plt.show()

rows = []
loss_rows = []
step_counter = 0
for name, log in stage_logs.items():
    for entry in log:
        r = entry.get("rewards/env_reward/mean", entry.get("reward"))
        if "loss" in entry:
            loss_rows.append({"stage": name, "global_step": step_counter + 1, "loss": entry["loss"]})
        if r is None:
            continue
        step_counter += 1
        rows.append({
            "stage": name,
            "global_step": step_counter,
            "env": r,
            "fmt":  entry.get("rewards/format_reward/mean", 0.0),
            "semantic": entry.get("rewards/semantic_action_reward/mean", 0.0),
            "idle": entry.get("rewards/anti_idle_reward/mean", 0.0),
        })

df = pd.DataFrame(rows)
df.to_csv(OUT / "reward_log.csv", index=False)

loss_df = pd.DataFrame(loss_rows)
if not loss_df.empty:
    plt.figure(figsize=(8, 4))
    for name, sub in loss_df.groupby("stage"):
        plt.plot(sub["global_step"], sub["loss"], label=f"stage {name}")
    plt.xlabel("global step"); plt.ylabel("loss")
    plt.title("Ghostexec SFT+GRPO — loss vs step")
    plt.legend(); plt.tight_layout()
    plt.savefig(OUT / "loss_curve.png", dpi=150); plt.show()

if not df.empty:
    plt.figure(figsize=(8, 4))
    for name, sub in df.groupby("stage"):
        plt.plot(sub["global_step"], sub["env"], label=f"stage {name}")
    plt.xlabel("global step"); plt.ylabel("mean env_reward")
    plt.title("Ghostexec GRPO — reward vs step (Unsloth)")
    plt.legend(); plt.tight_layout()
    plt.savefig(OUT / "reward_curve.png", dpi=150); plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(df["global_step"], df["env"],  label="env_reward")
    plt.plot(df["global_step"], df["fmt"],  label="format_reward")
    plt.plot(df["global_step"], df["semantic"], label="semantic_action_reward")
    plt.plot(df["global_step"], df["idle"], label="anti_idle_reward")
    plt.xlabel("global step"); plt.ylabel("mean component reward")
    plt.title("Reward components — hacking-watch")
    plt.legend(); plt.tight_layout()
    plt.savefig(OUT / "components.png", dpi=150); plt.show()
else:
    print("No numeric reward log found — skipping curve plots.")

In [ ]:
final_adapter = OUT / "adapter_final"
policy.save_pretrained(final_adapter)
tokenizer.save_pretrained(final_adapter)
print("final adapter →", final_adapter)

if HUB_REPO_ID and os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi, login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    policy.push_to_hub(HUB_REPO_ID, commit_message=f"ghostexec GRPO adapter ({RUN_NAME})")
    tokenizer.push_to_hub(HUB_REPO_ID)
    api = HfApi()
    for fname in ("reward_log.csv", "before_after.png", "reward_curve.png", "components.png"):
        p = OUT / fname
        if p.exists():
            api.upload_file(path_or_fileobj=str(p), path_in_repo=fname, repo_id=HUB_REPO_ID)
    print("pushed adapter + artefacts →", HUB_REPO_ID)
else:
    print("HUB_REPO_ID / HF_TOKEN not set — skipping push.")

In [ ]:
manifest = {
    "env_url":    GHOSTEXEC_ENV_URL,
    "model":      MODEL_ID,
    "run":        RUN_NAME,
    "stack":      {"unsloth": True, "trl": "0.22.2", "pipeline": "SFT->GRPO"},
    "rewards": {
        "random_mean":  summary["random"],
        "frozen_mean":  summary["frozen"],
        "trained_mean": summary["trained"],
        "improvement_vs_frozen": summary["trained"] - summary["frozen"],
    },
    "stages":       ["SFT"] + list(stage_logs.keys()),
    "reward_fns":   ["env_reward", "format_reward", "semantic_action_reward", "anti_idle_reward"],
    "curriculum":   {
        "type": "easy_to_full",
        "stage_scales": {
            "B": {"env": 0.85, "local": 0.60},
            "C": {"env": 0.95, "local": 0.40},
            "D": {"env": 1.00, "local": 0.25},
        },
    },
    "adapter_path": str(final_adapter),
    "mean_space_latency_ms": round(sum(env.latency_ms) / max(len(env.latency_ms), 1), 1),
    "n_space_calls":         len(env.latency_ms),
}
print(json.dumps(manifest, indent=2))
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("\nmanifest →", OUT / "manifest.json")